# 06 · Bit Manipulation

Computers store everything as bits (0 and 1). Bit tricks let us solve some problems faster and with less memory.

**Topics covered**
1. Decimal ⇄ binary conversion
2. How integers are stored: one's and two's complement
3. Bitwise operators (`&`, `|`, `^`, `~`, `<<`, `>>`)
4. Swapping two numbers with XOR
5. Minimum bit flips to convert one number into another
6. Single number (the element that appears once)
7. Power set using bitmasks

## 1. Decimal ⇄ binary

### Decimal → binary
Repeatedly divide by 2 and note the **remainders**. The remainders read **bottom-to-top** give the binary number — that's why we reverse the list at the end.

```
13 ÷ 2 = 6  r 1
 6 ÷ 2 = 3  r 0
 3 ÷ 2 = 1  r 1
 1 ÷ 2 = 0  r 1     → read upward: 1101
```

> 🐞 **Fix:** the earlier version returned a *list* of characters and returned an empty result for `0`. Now it returns a string and handles `0`.

In [1]:
def convert_to_binary(num: int) -> str:
    if num == 0:
        return "0"
    result = []
    while num > 0:
        result.append("1" if num % 2 == 1 else "0")   # remainder
        num //= 2
    return "".join(reversed(result))                   # remainders come out backwards

print(convert_to_binary(13))                     # 1101
print(convert_to_binary(0))                      # 0
print(convert_to_binary(10), "==", bin(10)[2:])  # built-in check

1101
0
1010 == 1010


### Binary → decimal
Each bit is worth a power of 2. Scan from the **rightmost** bit (power 0) to the left and add `bit × 2^power`.

`1101 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 8 + 4 + 0 + 1 = 13`

In [2]:
def convert_to_decimal(binary: str) -> int:
    decimal = 0
    power = 0
    for index in range(len(binary) - 1, -1, -1):       # right → left
        decimal += int(binary[index]) * (2 ** power)
        power += 1
    return decimal

print(convert_to_decimal("1101"))                # 13
print(convert_to_decimal("1010"), "==", int("1010", 2))

13
10 == 10


## 2. How a computer stores integers

**One's complement:** flip every bit (0 → 1, 1 → 0).
`1101 → 0010`

**Two's complement:** one's complement **+ 1**. Computers use this to store negative numbers.
`1101 → 0010 → 0011`

In a 32-bit integer the **leftmost (32nd) bit is the sign bit**: `0` means positive, `1` means negative.

**To find the value of `~x`** (NOT operator): flip all bits; if the result is negative (sign bit is 1), take its two's complement to read the magnitude. The shortcut: **`~x == -(x + 1)`**.

## 3. Bitwise operators

| Operator | Name | Rule | Example |
|---|---|---|---|
| `a & b` | AND | bit is 1 only if **both** bits are 1 | `13 & 7 = 5` |
| `a \| b` | OR | bit is 1 if **at least one** bit is 1 | `13 \| 7 = 15` |
| `a ^ b` | XOR | bit is 1 if the bits are **different** | `13 ^ 7 = 10` |
| `~a` | NOT | flips all bits (`-(a+1)`) | `~13 = -14` |
| `a << k` | Left shift | multiplies by 2ᵏ | `13 << 1 = 26` |
| `a >> k` | Right shift | divides by 2ᵏ (floor) | `13 >> 1 = 6` |

**XOR properties (very useful):**
- `x ^ x = 0` (same number cancels out)
- `x ^ 0 = x` (XOR with zero changes nothing)
- XOR of many bits is `1` when the count of 1s is **odd**, `0` when it is **even**
- XOR is commutative and associative, so order doesn't matter

**Shifts:** `x >> k = x // 2**k` and `x << k = x * 2**k`.
Example: `13 = 1101`; `13 >> 1 = 0110 = 6`; `13 >> 3 = 0001 = 1`; `13 << 1 = 11010 = 26`; `13 << 3 = 1101000 = 104`.

In [3]:
a, b = 13, 7
print("13 & 7  =", a & b, "  ", bin(a), "&", bin(b), "=", bin(a & b))
print("13 | 7  =", a | b)
print("13 ^ 7  =", a ^ b)
print("~13     =", ~a, " (= -(13 + 1))")
print("13 >> 1 =", a >> 1, "  13 >> 3 =", a >> 3, "  13 >> 4 =", a >> 4)
print("13 << 1 =", a << 1, " 13 << 3 =", a << 3)
print("13 ^ 13 =", 13 ^ 13, "  13 ^ 0 =", 13 ^ 0)

13 & 7  = 5    0b1101 & 0b111 = 0b101
13 | 7  = 15
13 ^ 7  = 10
~13     = -14  (= -(13 + 1))
13 >> 1 = 6   13 >> 3 = 1   13 >> 4 = 0
13 << 1 = 26  13 << 3 = 104
13 ^ 13 = 0   13 ^ 0 = 13


## 4. Swap two numbers without a temporary variable

Using XOR three times:
```
a = a ^ b      # a now holds (a⊕b)
b = a ^ b      # (a⊕b)⊕b = a   → b becomes the ORIGINAL a
a = a ^ b      # (a⊕b)⊕a = b   → a becomes the ORIGINAL b
```
This works because `x ^ x = 0` and `x ^ 0 = x`.

In [4]:
a, b = 5, 10
print("before:", a, b)
a = a ^ b
b = a ^ b
a = a ^ b
print("after :", a, b)

before: 5 10
after : 10 5


## 5. Minimum bit flips to convert `start` into `goal`

`start ^ goal` has a `1` in exactly the positions where the two numbers **differ**. So the answer = **number of 1-bits** in `start ^ goal` (also called the *popcount* or Hamming distance).

Three ways to count 1-bits:
1. Check each of the 32 positions with `x & (1 << i)`
2. Python shortcut: `bin(x).count("1")`
3. **Brian Kernighan's trick:** `x & (x - 1)` removes the lowest set bit. Repeat until `x` is 0 — loops only as many times as there are 1-bits.

In [5]:
start, goal = 10, 7                       # 1010 and 0111
diff = start ^ goal                       # 1101 → three positions differ

# Method 1: test every bit position
count = 0
for i in range(32):
    if (diff & (1 << i)) != 0:            # is bit i set?
        count += 1
print("Check each bit :", count)

# Method 2: built-in
print("bin().count    :", bin(diff).count("1"))

# Method 3: Kernighan
x, count = diff, 0
while x:
    x &= x - 1                            # clear the lowest set bit
    count += 1
print("Kernighan      :", count)

Check each bit : 3
bin().count    : 3
Kernighan      : 3


## 6. Single number: the element that appears once

All other numbers appear **exactly twice**.

| Approach | Time | Space |
|---|---|---|
| Frequency map (dict) | O(n) | O(n) |
| **XOR everything** | O(n) | **O(1)** |

XOR all elements: pairs cancel (`x ^ x = 0`) and the lone element survives (`x ^ 0 = x`).

In [6]:
# Hash map approach
arr = [1, 2, 2, 3, 3, 4, 1]
freq = {}
for x in arr:
    freq[x] = freq.get(x, 0) + 1
print("Hash map:", [k for k, v in freq.items() if v == 1])

# XOR approach
arr = [11, 23, 32, 32, 23, 24, 45, 45, 24]
ans = 0
for x in arr:
    ans ^= x
print("XOR     :", ans)                   # 11

Hash map: [4]
XOR     : 11


## 7. Power set using bitmasks

A set with `n` elements has **2ⁿ subsets**. Each number from `0` to `2ⁿ − 1` can act as a *mask*: if bit `i` is `1`, include `nums[i]` in the subset.

```
mask 0 (000) → []
mask 1 (001) → [1]
mask 5 (101) → [1, 3]
mask 7 (111) → [1, 2, 3]
```
`1 << n` is a fast way to write `2**n`.

**Time:** O(n · 2ⁿ)

In [7]:
nums = [1, 2, 3]
n = len(nums)
total_subsets = 1 << n                    # 2**n
result = []
for mask in range(total_subsets):
    subset = []                           # (renamed from `list` to avoid shadowing the built-in)
    for i in range(n):
        if mask & (1 << i):               # is bit i set in this mask?
            subset.append(nums[i])
    result.append(subset)
print(result)

[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


---
### ✅ Key takeaways
- `x ^ x = 0`, `x ^ 0 = x` → find the single / missing number, swap without temp.
- `x & (x - 1)` clears the lowest set bit (counting bits, power-of-two check: `x & (x-1) == 0`).
- `1 << n` = `2**n` and bitmasks enumerate all subsets.
- Python ints have unlimited size, so `~x` is simply `-(x+1)` (no fixed 32-bit wraparound).